# Notebook de Simulação e Teste para o LiveTrader

Este notebook permite executar o fluxo de trabalho do `LiveTrader` passo a passo, ideal para depuração e simulações rápidas.

**Pré-requisitos:**
1. O terminal MetaTrader 5 deve estar aberto e logado.
2. Um modelo de produção (`prod_model.keras`) e um scaler (`prod_scaler.joblib`) devem ter sido gerados pelo script `train_model.py` e estar na pasta `models/`.

In [7]:
import pandas as pd
import yaml
import sys
import importlib
from pathlib import Path

# Adiciona a pasta 'src' ao path para permitir as importações dos nossos módulos
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Cria o caminho absoluto para o arquivo de configuração
config_file_path = project_root / 'configs' / 'main.yaml'
    
from src.data_handler.provider import YFinanceProvider, MetaTraderProvider
from src.strategies.sentiment_lstm import SentimentLSTMStrategy
from src.strategies.lstm import LSTMStrategy

# Importa a classe LiveTrader que vamos testar
from src.live_trader import LiveTrader

In [16]:
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

config_path="configs/main.yaml"

absolute_config_path = project_root / config_path

try:
    with open(absolute_config_path, 'r') as file:
        config = yaml.safe_load(file)
except FileNotFoundError:
    # Fallback para quando executado de um notebook
    print("Arquivo de configuração não encontrado. Tentando caminho alternativo...")
    config_file_path = Path.cwd() / 'configs' / 'main.yaml'

strategy_name = config['backtest_settings']['strategy_name']
module_path = f"src.strategies.{config['backtest_settings']['strategy_module']}"
strategy_module = importlib.import_module(module_path)
print("Buscando Classe:", strategy_name)
print("Módulo Importado:", strategy_module)

StrategyClass = getattr(strategy_module, strategy_name)

print("Classe Encontrada:", StrategyClass)

Buscando Classe: LSTMStrategy
Módulo Importado: <module 'src.strategies.lstm' from 'c:\\projects\\wtnps-trade\\src\\strategies\\lstm.py'>
Classe Encontrada: <class 'src.strategies.lstm.LSTMStrategy'>


### Passo 1: Instanciar e Inicializar o Trader

Esta célula cria uma instância do `LiveTrader` e chama o método `.initialize()`, que conecta ao MT5 e carrega o modelo salvo.

In [13]:
# Cria a instância do trader
trader = LiveTrader(config_path=config_file_path)

# Inicializa (conecta ao MT5 e carrega o modelo)
is_initialized = trader.initialize()

if is_initialized:
    print(f"Trader inicializado com sucesso para o ativo: {trader.ticker}")
else:
    print("Falha ao inicializar o trader. Verifique a conexão com o MetaTrader 5.")

2025-10-07 15:11:08,955 - INFO - Diretório de cache de dados inicializado em: C:\projects\wtnps-trade\notebooks\.cache_data
2025-10-07 15:11:12,863 - INFO - Inicializando o robô trader...
2025-10-07 15:11:15,269 - INFO - Carregando modelo de models/prod_model.keras e scaler de models/prod_scaler.joblib


ValueError: File not found: filepath=models/prod_model.keras. Please ensure the file is an accessible `.keras` zip file.

### Passo 2: Executar um Único Ciclo de Decisão (Single Tick)

Esta célula executa o corpo do loop `while` do robô uma única vez. Você pode executar esta célula repetidamente para simular a passagem do tempo candle a candle.

In [6]:
def run_single_tick(trader_instance):
    """Executa um ciclo completo de busca de dados, predição e decisão."""
    if not trader_instance.model:
        print("Trader não inicializado. Execute a célula anterior primeiro.")
        return
    
    print("--- Executando um novo ciclo de decisão ---")
    # 1. Buscar dados recentes
    print(f"Buscando os 300 candles mais recentes de {trader_instance.ticker}...")
    latest_data = trader_instance.provider.get_latest_rates(
        trader_instance.ticker, 
        300, 
        trader_instance.live_config['timeframe']
    )
    
    # se não localizou pelo provider, tenta pelo MetaTraderProvider diretamente
    if latest_data.empty:
        import MetaTrader5 as mt5
        if not mt5.initialize():
            print("Falha ao conectar ao MetaTrader 5")
            return
        
        rates = mt5.copy_rates_from_pos(trader_instance.ticker, mt5.TIMEFRAME_H1, 0, 300)
        latest_data = pd.DataFrame(rates)
        latest_data['time'] = pd.to_datetime(latest_data['time'], unit='s')
        latest_data.set_index('time', inplace=True)
        latest_data.rename(columns={'tick_volume': 'volume'}, inplace=True)

        mt5.shutdown()  # encerra a conexão com o MT5
    
    print(f"Último candle recebido: {latest_data.index[-1]}")
    display(latest_data.tail(3))

    # 2. Gerar features
    print("\nGerando features com os dados recentes...")
    featured_data = trader_instance.strategy.define_features(latest_data)
    X_live = featured_data[trader_instance.strategy.get_feature_names()].dropna()
    
    if X_live.empty:
        print("Não há dados suficientes para gerar features.")
        return

    # 3. Fazer a previsão (sinal)
    print("\nGerando sinal com o modelo de IA...")
    signal = trader_instance.model.predict(X_live)[-1] # Pega a última predição
    signal_text = 'COMPRA' if signal == 1 else 'VENDA'
    print(f"==> SINAL GERADO: {signal_text} ({signal}) ==<")

    # 4. Lógica de decisão (sugestão)
    print("\nAplicando lógica de decisão...")
    if trader_instance.current_position is None:
        if signal == 1:
            trader_instance._execute_trade('BUY')
        elif signal == 0:
            trader_instance._execute_trade('SELL')
    else:
        print(f"Já existe uma posição aberta ({trader_instance.current_position}). Nenhuma nova ordem será enviada.")
        
    print("--- Ciclo de decisão concluído ---")

# Executa a função para o nosso trader
if is_initialized:
    run_single_tick(trader)

NameError: name 'is_initialized' is not defined

### Passo 3: Encerrar a Conexão

Ao final dos seus testes, execute esta célula para garantir que a conexão com o MetaTrader 5 seja encerrada corretamente.

In [11]:
import MetaTrader5 as mt5

print("Encerrando conexão com o MetaTrader 5...")
mt5.shutdown()
print("Conexão encerrada.")

Encerrando conexão com o MetaTrader 5...
Conexão encerrada.
